# Truthfulness vector extraction

Capture the last prompt row at layers 14–27, then reuse each fold’s capture for DiffMean, exact centered PCA and linear probe. The two original question folds, prompt formatting and extraction settings are retained. Capture uses the default 256 MiB raw-data budget; exact extraction has an explicit 1 GiB working-allocation budget. These budgets exclude model weights, KV cache and library overhead.

Set `CUDA_VISIBLE_DEVICES` and `EASYSTEER_TP` before running to use multiple GPUs. Capture supports compiled execution; eager mode is not required. Updated cells have not been rerun at full experiment scale.


In [ ]:
import pandas as pd

df = pd.read_csv("TruthfulQA.csv")

types = df["Type"].tolist()
categories = df["Category"].tolist()
questions = df["Question"].tolist()
best_answers = df["Best Answer"].tolist()
correct_answers = df["Correct Answers"].tolist()
incorrect_answers = df["Incorrect Answers"].tolist()
sources = df["Source"].tolist()
correct_answers = [c.split("; ") for c in correct_answers]
incorrect_answers = [ic.split("; ") for ic in incorrect_answers]
print(best_answers[0])
print(correct_answers[0]) 
print(incorrect_answers[0])
print(len(questions))

In [ ]:
import os

from vllm import LLM
from vllm.capture import SelectSpec

from easysteer.capture import capture, release_capture_cache
from easysteer.extraction import extract

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
llm = LLM(
    model=os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen2.5-1.5B-Instruct"),
    tensor_parallel_size=int(os.environ.get("EASYSTEER_TP", "1")),
)
# Match the layers used by steer.ipynb; capture has its own 256 MiB budget.
LAYERS = list(range(14, 28))
# Exact PCA and probe reuse one capture and preflight their working allocations.
EXTRACTION_BYTES = 1024**3


In [ ]:
positive = []
for i in range(408):
    for j in range(len(correct_answers[i])):
        test_str = f"""<|im_start|>user
{questions[i]}
<|im_end|>\n<|im_start|>assistant
{correct_answers[i][j]}.
"""
        positive.append(test_str)


negative = []
for i in range(408):
    for j in range(len(incorrect_answers[i])):
        test_str = f"""<|im_start|>user
{questions[i]}
<|im_end|>\n<|im_start|>assistant
{incorrect_answers[i][j]}.
"""
        negative.append(test_str)

In [ ]:
labels = [True] * len(positive) + [False] * len(negative)
captured = capture(
    llm,
    positive + negative,
    layers=LAYERS,
    select=SelectSpec(prompt_positions=[-1]),
    max_tokens=1,
    steering=False,
)


In [ ]:
# Keep the original centered PCA and linear probe; do not substitute
# approximate incremental PCA, which would change the experiment.
for name, method, options in (
    ("caa", "diffmean", {}),
    ("pca", "pca", {"variant": "center"}),
    ("probe", "linear_probe", {}),
):
    control_vector = extract(
        captured,
        labels,
        method=method,
        token_pos=-1,
        normalize=True,
        max_working_bytes=EXTRACTION_BYTES,
        **options,
    )
    control_vector.export_gguf(f"real1-{name}.gguf")
del captured
release_capture_cache(llm)


In [ ]:
positive = []
for i in range(408,817):
    for j in range(len(correct_answers[i])):
        test_str = f"""<|im_start|>user
{questions[i]}
<|im_end|>\n<|im_start|>assistant
{correct_answers[i][j]}
"""
        positive.append(test_str)


negative = []
for i in range(408,817):
    for j in range(len(incorrect_answers[i])):
        test_str = f"""<|im_start|>user
{questions[i]}
<|im_end|>\n<|im_start|>assistant
{incorrect_answers[i][j]}
"""
        negative.append(test_str)

In [ ]:
labels = [True] * len(positive) + [False] * len(negative)
captured = capture(
    llm,
    positive + negative,
    layers=LAYERS,
    select=SelectSpec(prompt_positions=[-1]),
    max_tokens=1,
    steering=False,
)


In [ ]:
# Keep the original centered PCA and linear probe; do not substitute
# approximate incremental PCA, which would change the experiment.
for name, method, options in (
    ("caa", "diffmean", {}),
    ("pca", "pca", {"variant": "center"}),
    ("probe", "linear_probe", {}),
):
    control_vector = extract(
        captured,
        labels,
        method=method,
        token_pos=-1,
        normalize=True,
        max_working_bytes=EXTRACTION_BYTES,
        **options,
    )
    control_vector.export_gguf(f"real2-{name}.gguf")
del captured
release_capture_cache(llm)
